# 10 RAG项目方案讨论与初版代码生成


## 1. 对应课程与本篇范围

前几篇用记账桌面 APP 练习了需求表达、上下文管理、Skills、Agent 和 Hooks。本篇开始一个新的工程：**基于 LangChain 的电商商品知识库问答系统**。重点是先讨论方案，再让 AI 生成能够继续验证的初版。

对应网盘视频：

```text
21-RAG项目-方案讨论和初版代码生成.mp4
```

课程见 [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)，提取码 `1234`。配套讲义为 [Part3 LangChain RAG项目实战](https://www.yuque.com/xxcls/vibecoding/mtwqfc61tifo9cgi)。原项目位于网盘 `项目/LangChainRAG项目/`。

讲义给出了完整的项目初始化提示词。本篇先保留这段需求，再结合已读取的项目 `README.md`、依赖清单、配置、数据模型和 RAG 核心源码解释方案。**讲义没有提供完整的方案讨论对话；项目文件是共享快照，不能据此还原第 21 节每一步的代码版本。**文中标为“整理”或“补充”的提示词与操作，是帮助理解当前环节的内容。

本篇不展开后续第 22 节的单元测试与 BUG 调试、第 23 节的技能和 Agent 迁移及 Git 提交、第 24 节的并发压力测试。视频未逐段观看或转录，也没有在本次整理中启动实际项目。


## 2. 原始提示词：先把项目目标和约束说清楚

讲义以毕业设计为背景，让 AI 协助设计一个浏览器里的电商问答系统。以下保留原提示词的内容和顺序：

```text
我的毕设是，通过langchain框架，开发RAG企业级知识库问答系统，用户可以通过浏览器进行知识库问答操作；
问答系统主要是针对电商平台售卖商品进行回答，提供的知识库多数是商品相关信息，帮助更好的回答用户关于商品的问题；
技术选型上，我听说AI领域langchain是非常知名的开发框架，所以必须选择langchain，其余技术栈你来决定；

我需要有如下功能：
1. 能够支持用户在浏览器中进行知识库管理
2. 能够支持用户在浏览器中和模型进行知识库问答，问答的时候要尽量引用知识库内容作为参考，在回答中能够显示引用的知识库片段是哪些
3. 能够支持多用户多会话管理，每个用户有每个用户的独立会话
4. 能够保持用户的会话记录，用户不同时间段登录都能找回历史对话
5. 支持基础的用户注册登录功能，支持用户修改密码
6. 管理员用户名admin，密码123456，只有管理员能够打开知识库管理页面，进行知识库管理，其他用户只能进行知识库问答
7. 不仅仅实现基础功能，也要应用各类技术做性能优化，达到企业级使用的效果
8. 其余你觉得有必要可以添加的功能点

你帮我列出来方案计划，我们先讨论；
```

这段提示词包含四类信息：项目场景、必须使用的框架、可观察的功能要求、当前只讨论方案的工作阶段。

“必须使用 LangChain”是这次任务的约束；“其余技术栈你来决定”给 AI 留出了设计空间。最后一句“我们先讨论”限定了当前交付物：先说明技术方案和实施计划，再进入代码生成。它不等于已经确认模型返回的所有选型。

其中 `admin / 123456` 是课程的演示账号，后文也按原项目保留。它不是正式部署时应沿用的账号密码配置。


## 3. 把自然语言需求整理成可检查的功能

RAG 是 Retrieval-Augmented Generation，通常译为检索增强生成。本项目先从商品资料中检索相关片段，再把片段作为参考交给大模型生成回答。

用户的问题可能是“某耳机开启降噪能用多久”，知识库里则保存型号、续航、规格和价格等商品资料。框架需要把问题与相关资料接起来，应用还要处理登录、上传、会话和引用展示。

下表是按原需求整理的检查项：

| 原需求 | 方案中需要明确的内容 | 初版可观察的结果 |
|---|---|---|
| 浏览器管理知识库 | 上传入口、格式限制、索引状态、列表和删除 | 管理员上传后能看到文档处理状态 |
| 根据知识库回答 | 文档加载、分块、嵌入、检索、生成 | 能回答已上传资料中的商品问题 |
| 显示引用片段 | 给分块保留文件名等元数据，返回引用结构 | 回答旁能查看对应文件和片段 |
| 多用户、多会话 | 用户、会话、消息的所属关系 | 同一用户可切换会话，普通用户之间会话隔离 |
| 跨登录恢复历史 | 服务端持久化消息和引用 | 刷新或重新登录后能恢复历史 |
| 注册、登录、改密码 | 密码哈希、令牌、当前用户检查 | 能注册登录，并验证旧密码后修改密码 |
| 管理员专属管理 | 前端入口限制和后端权限检查 | 普通用户直接调用管理接口也被拒绝 |
| 性能优化 | 具体措施、适用范围、验证方法 | 有可核对的实现；吞吐量留待实际测试 |

“企业级”在提示词里是目标描述，不是已经通过的验收结论。方案应把它拆成具体行为和后续验证项，不能只凭项目名称或使用了某个框架就判断达标。

**补充：**原需求允许 AI 增加必要功能，但新增项仍应服务于当前主流程。项目快照中的流式显示、Markdown 渲染、文档分类和统计都可以解释其用途；不必因此扩展成另一套业务系统。


## 4. 方案讨论：先确定数据怎样流动

原资料没有完整保留 AI 的方案回复。下面依据原需求和项目快照，整理讨论时需要讲清楚的两条流程。

### 知识入库

管理员在浏览器上传商品文档，后端保存文件和文档记录，再进行加载、分块、嵌入和向量存储。完成后更新文档状态，浏览器才能告诉用户“已索引”。

```text
上传文档 → 保存文件与文档记录 → 加载文本 → 分块
         → 计算向量并写入 ChromaDB → 更新索引状态
```

### 知识问答

用户进入自己的会话并提问。后端检查身份和会话归属，读取近期历史，检索知识片段，组织模型输入，再把回答和引用返回浏览器并保存。

```text
用户问题 + 近期会话历史
         → 检索相关商品片段 → 组织参考文档与提示词
         → 大模型流式生成 → 返回正文与引用 → 保存消息
```

这两条流程对资料的使用方式不同：上传时处理文档，提问时检索已经建立的索引。首次问答验收前必须先完成文档入库，否则模型没有本项目的商品资料可用。

方案讨论还要区分两种存储：关系数据库负责用户、会话和文档状态；向量库负责相似片段检索。把全部数据都叫“知识库”，容易漏掉历史消息和引用的持久化设计。

**补充提示词：**可以在原始需求之后追问下列内容。这不是课堂对话原文。

```text
请先把方案讲清楚，暂时不要生成代码：
1. 列出技术栈及每项技术负责的功能。
2. 说明文档上传入库、知识问答两条完整流程。
3. 列出用户、会话、消息、知识文档的数据关系。
4. 说明管理员权限、会话归属和引用片段如何处理。
5. 列出目录结构、主要接口、配置项和分阶段实施顺序。
6. 把首版必须完成的功能与需要后续验证的性能目标分开列出。
有不确定的业务条件请明确提出，等方案确认后再开始实现。
```


## 5. 原项目采用的技术栈

以下选型来自共享项目 `README.md`，并已与相关配置、依赖或源码核对。

| 层级 | 原项目技术 | 负责的事情 |
|---|---|---|
| RAG 框架 | LangChain、LangChain-Community | 文档、分块、模型消息、检索组件的组织 |
| 大语言模型 | 百炼 DashScope，`qwen-plus` | 根据参考文档生成中文回答 |
| 嵌入模型 | 百炼 `text-embedding-v2` | 将文档和问题转为可比较的向量 |
| 后端 | Python、FastAPI、Uvicorn | 登录、上传、会话与问答接口 |
| 数据校验与配置 | Pydantic、pydantic-settings | 请求数据结构与 `.env` 配置 |
| 关系存储 | SQLite、SQLAlchemy 2.0、aiosqlite | 保存账号、会话、消息和文档记录 |
| 向量存储 | ChromaDB、langchain-chroma | 持久化片段向量并进行相似检索 |
| 身份认证 | JWT、bcrypt | 登录令牌、密码哈希及访问检查 |
| 前端 | React 18、TypeScript、Vite | 浏览器页面与开发构建 |
| 页面组件 | Ant Design 5 | 表单、列表和管理界面组件 |
| 前端状态与请求 | Zustand、Axios | 应用状态和 HTTP 请求基础 |
| 回答展示 | SSE、react-markdown | 逐步显示回答和渲染 Markdown |

嵌入模型与大语言模型承担不同职责。前者把文本变成向量以便检索；后者接收问题和参考资料并生成文字。项目中这两条调用路径也不同，不能只配置对话模型就假定文档嵌入已经可用。

SQLite 保存业务记录，ChromaDB 保存检索索引；本快照采用本地持久化目录。方案讨论中可以说明这种选择如何降低初版运行成本，但不能把本地文件存储解释成已经具备分布式部署能力。


## 6. 初版工程应生成哪些文件

共享项目的主要结构如下。目录树只列出与本篇相关的源码和配置；共享目录还带有 `venv/`、`node_modules/`、测试及构建产物，它们不等于需要重新生成的业务代码。

```text
LangChainRAG项目/
├── backend/
│   ├── app/
│   │   ├── main.py                 # 应用入口、生命周期、路由和 CORS
│   │   ├── config.py               # 模型、数据库、RAG 与上传参数
│   │   ├── database.py             # 异步数据库会话
│   │   ├── api/                    # auth、conversations、chat、kb、system
│   │   ├── core/                   # security、dependencies
│   │   ├── models/                 # user、conversation、message、knowledge_document
│   │   ├── schemas/                # 请求与响应数据结构
│   │   ├── services/               # 认证、会话、知识库和 RAG 编排
│   │   └── rag/
│   │       ├── loader.py
│   │       ├── splitter.py
│   │       ├── embeddings.py
│   │       ├── vector_store.py
│   │       ├── retriever.py
│   │       └── chain.py
│   ├── .env.example
│   ├── .env
│   ├── requirements.txt
│   └── data/                       # 后端运行时数据
├── frontend/
│   ├── src/
│   │   ├── api/
│   │   ├── components/
│   │   ├── hooks/
│   │   ├── pages/
│   │   ├── store/
│   │   ├── types/
│   │   ├── utils/
│   │   ├── App.tsx
│   │   └── main.tsx
│   └── package.json
├── data/products/
│   ├── clothing.md
│   └── electronics.md
├── README.md
└── start.bat
```

`api/` 接收 HTTP 请求，`services/` 组织业务流程，`rag/` 实现加载、分割、检索和生成。这样读代码时可以沿一次请求向下追踪，避免把登录、上传和模型调用都混进一个入口文件。

`main.py` 在启动生命周期中调用 `init_db()`，随后调用 `seed_admin()` 并提交管理员数据；接着由各路由模块提供接口。它还根据 `CORS_ORIGINS` 设置跨源访问规则，让独立端口上的浏览器前端能够访问后端。

注意两个 `data/` 的含义：项目根目录的 `data/products/` 是示例商品资料；从 `backend/` 启动时，`./data/app.db`、`./data/chromadb`、`./data/uploads` 才是后端使用的运行数据位置。


## 7. 用户、会话、消息与知识文档的数据关系

原项目在 `backend/app/models/` 定义四个主要模型：

| 模型 / 表 | 关键字段 | 作用 |
|---|---|---|
| `User` / `users` | `id`、`username`、`password_hash`、`role`、`is_active` | 账号与角色，用户名唯一 |
| `Conversation` / `conversations` | `id`、`user_id`、`title`、`is_active` | 用户拥有的独立会话 |
| `Message` / `messages` | `conversation_id`、`role`、`content`、`sources`、`token_count` | 会话里的消息和回答引用 |
| `KnowledgeDocument` / `knowledge_documents` | `filename`、`file_type`、`file_size`、`chunk_count`、`status`、`uploaded_by` | 上传文件的管理和索引状态 |

一个用户可以拥有多个会话，一个会话可以拥有多条消息；知识文档通过 `uploaded_by` 记录上传者。消息的 `sources` 使用文本字段保存 JSON 字符串，用于恢复历史回答时继续显示引用。

这里的“多会话”是业务会话：不同会话有各自的消息列表和上下文。JWT 用来识别当前登录用户，它与对话会话的编号和消息历史承担不同职责。

`database.py` 使用 SQLAlchemy 异步会话。`get_db()` 为请求提供数据库会话，正常结束时提交，异常时回滚；`init_db()` 使用 `Base.metadata.create_all` 创建表。快照还配置了 SQLite 的 WAL 模式和锁等待参数，这些是具体实现措施，不是并发测试结果。

会话删除采用 `is_active = False` 的软删除方式，列表查询过滤活动会话。**列表中隐藏了一个会话，不代表所有按 ID 访问的路径都自动增加了同样的过滤条件**；实际行为仍应按对应接口检查。本篇先理解数据关系，后续再进入完整测试和调试。


## 8. 权限要落在后端，不能只隐藏管理入口

原项目的认证流程位于 `services/auth_service.py`：注册用户固定为 `role="user"`，密码通过哈希保存；登录时验证密码和账号状态，再创建包含用户 ID 的 JWT。修改密码需要先验证旧密码。

`core/dependencies.py` 的 `get_current_user()` 从 `Authorization: Bearer ...` 中解析令牌，解码后读取 `sub` 用户 ID，再查询数据库中的用户和启用状态。管理员依赖继续检查数据库用户的角色。

下面是原项目 `get_admin_user()`：

```python
async def get_admin_user(
    current_user: User = Depends(get_current_user),
) -> User:
    """仅允许 admin 角色访问"""
    if current_user.role != "admin":
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="需要管理员权限",
        )
    return current_user
```

知识库路由通过 `Depends(get_admin_user)` 限制访问。前端隐藏菜单可以改善使用体验，真正阻止普通用户管理文档的是后端检查。原需求写“只有管理员”，源码实现依据 `role == "admin"`，不是在每个接口中比较用户名字符串。

会话还要检查归属。例如聊天接口在调用模型之前比较 `conv.user_id` 与 `current_user.id`，不相等返回 `403`。会话列表也按当前用户过滤。

**源码差异：**`conversation_service.py` 的详情查询允许会话所有者或管理员读取；发送消息、删除会话、改标题则要求所有者。因此不能仅照 README 中“所有者”的概括，写成“所有会话操作都只有所有者可执行”。这一例外需要在实际验收时明确。

管理员种子数据也有一个容易忽略的行为：`seed_admin()` 不仅在账号不存在时创建管理员；已有账号时，还会用 `settings.ADMIN_PASSWORD` 重新生成密码哈希。当前快照中，管理员在页面修改密码后再重启服务，可能被恢复为配置中的密码。函数注释“仅在不存在时创建”没有完整表达这段行为，应以函数体为准。


## 9. 知识入库：上传成功与索引完成是两个阶段

`backend/app/api/kb.py` 的实际上传接口是：

```text
POST /api/kb/documents/upload
```

它接收 `multipart/form-data`，文件字段为 `file`，可选商品分类字段为 `product_category`。路由先检查管理员身份，再验证扩展名和大小，把文件保存到 `./data/uploads`。文件名增加 UUID 前缀，数据库记录也使用保存后的名字。

默认允许 `txt,md,pdf,docx,csv,xlsx`，上传上限为 `50 MB`。上传后创建 `status="processing"`、`chunk_count=0` 的记录，并通过 FastAPI `BackgroundTasks` 安排 `_background_ingest()`；返回文件记录时，文档还可能在处理。

`services/rag_service.py` 的 `ingest_document()` 顺序为：

1. `load_document(file_path)` 加载文件。
2. `splitter.split_documents(raw_docs)` 切分文本。
3. 给分块增加文件和业务元数据。
4. 分批调用 `vectorstore.add_documents(batch)` 写入 ChromaDB。
5. 设置文档 `status="indexed"` 和 `chunk_count`，由调用者提交事务。

下面摘录该函数内部的元数据处理，保留原缩进：

```python
        # 3. 元数据丰富
        for i, chunk in enumerate(chunks):
            chunk.metadata.update({
                "source_file": file_path,
                "filename": filename,
                "chunk_index": i,
                "total_chunks": len(chunks),
                "product_category": product_category or "",
                "chunk_hash": _compute_chunk_hash(chunk.page_content),
                "kb_doc_id": kb_doc_id,
            })
```

`filename`、`chunk_index` 用于引用展示，`kb_doc_id` 把向量片段关联回文档记录，也用于按文档删除向量。`chunk_hash` 虽然写进了元数据，但该函数没有据此查询或跳过重复块，不能把“保存哈希”写成已经完成去重。

**状态边界：**函数异常分支尝试写入 `failed` 和错误信息后重新抛出异常；后台调用者随即执行 `rollback()`。所以不能只看异常分支，就断言失败状态一定成功持久化。验收时如果文档长期停在 `processing`，应结合日志和事务行为定位，不能把上传接口的 `201` 当作索引完成。


## 10. 文档加载与中文分块

`rag/loader.py` 根据扩展名选择加载器：

| 文件 | 加载器 |
|---|---|
| `.txt`、`.md` | `TextLoader`，UTF-8 |
| `.pdf` | `PyPDFLoader` |
| `.csv` | `CSVLoader`，UTF-8 |
| `.xlsx` | `UnstructuredExcelLoader`，`mode="elements"` |
| `.docx` | `UnstructuredWordDocumentLoader` |

加载器产出 LangChain 的 `Document`，其中 `page_content` 是文本，`metadata` 保存来源等信息。后续切分和检索继续携带这些元数据。

`rag/splitter.py` 的核心代码如下：

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter
from app.config import settings


def get_text_splitter() -> RecursiveCharacterTextSplitter:
    """获取中文感知的文本分割器"""
    return RecursiveCharacterTextSplitter(
        chunk_size=settings.CHUNK_SIZE,
        chunk_overlap=settings.CHUNK_OVERLAP,
        # 中文感知分隔符：段落 → 换行 → 句子结束 → 分号 → 逗号 → 空格 → 字符
        separators=["\n\n", "\n", "。", "；", "，", " ", ""],
        length_function=len,
    )
```

默认 `CHUNK_SIZE=800`、`CHUNK_OVERLAP=100`。分隔符从段落、换行到中文句号等逐步细化，在满足长度条件的同时尽量保留语义边界。重叠部分用于减少相邻块边界造成的信息割裂。

这里使用 `length_function=len`，长度按 Python 字符串长度计算，**不是模型 token 数量**。参数是当前项目的默认值，需要结合商品文档和检索效果验证，不能当作所有 RAG 项目通用的最佳值。

先选择短小的 UTF-8 Markdown 商品资料完成入库，再检查 PDF、Word、Excel 等加载路径，可以把“主流程可用”与“每种格式都能处理”分别确认。允许扩展名也不等于相应解析依赖已经齐全。


## 11. 嵌入与检索：选出哪些片段交给模型

`rag/embeddings.py` 自定义 `DashScopeEmbeddings(Embeddings)`：

- `embed_documents()` 遍历文本，使用 `dashscope.TextEmbedding.call()` 获取文档向量。
- `embed_query()` 使用相同嵌入模型获取问题向量。
- 调用非成功状态会抛出 `RuntimeError`。

`rag/vector_store.py` 使用 `Chroma`，配置集合名 `ecommerce_kb`、嵌入实现和持久化路径 `./data/chromadb`。`get_vectorstore()` 懒加载并复用进程内实例。

摄入函数把分块按每组 `25` 个交给向量库；但当前自定义 `embed_documents()` 内部仍逐条发起嵌入请求。因此这份源码能够证明“分组写入”，不能证明已经做到“一次 API 请求批量嵌入 25 条”。

问答链调用 `retrieve_with_scores(question)`，检索过程为：

1. 从 ChromaDB 执行 `similarity_search_with_relevance_scores`。
2. 默认取前 `RETRIEVAL_TOP_K=5` 个候选。
3. 保留相关分数不低于 `RETRIEVAL_SCORE_THRESHOLD=0.4` 的片段。
4. 返回 `(Document, score)` 列表，供上下文和引用共同使用。

`0.4` 是这份实现中的过滤阈值，不是“答案正确概率 40%”。检索结果可能少于 5 条，也可能为空；仍需要用实际商品问题判断片段是否相关。

快照把同步检索放到 `asyncio.to_thread()`，并用 `asyncio.Semaphore(10)` 限制同时进入的检索数。它们说明代码如何安排并发，不代表端到端问答已达到某个并发量或响应时间，本篇也不把它们归因于第 21 节初版时就已存在。

**调用边界：**管理端 `/api/kb/search` 支持品类参数；当前问答链使用的 `retrieve_with_scores()` 没有品类参数。不能因为某个函数或界面存在品类字段，就写成所有问答都已经按品类过滤。该检索函数还会捕获异常并返回空列表，排查空引用时应同时检查检索日志。


## 12. 回答生成：把商品资料放进模型输入

`rag/chain.py` 使用 `langchain_openai.ChatOpenAI` 连接百炼的兼容接口。这里的客户端类名不代表调用的是 OpenAI 模型；实际服务由 `base_url` 和模型配置决定。

原项目的模型构造函数：

```python
def get_llm() -> ChatOpenAI:
    """ChatOpenAI -> 百炼 MaaS 业务空间端点"""
    return ChatOpenAI(
        model=settings.LLM_MODEL,
        api_key=settings.DASHSCOPE_API_KEY,
        base_url=settings.DASHSCOPE_API_BASE,
        temperature=settings.LLM_TEMPERATURE,
        streaming=True,
    )
```

默认模型为 `qwen-plus`，温度为 `0.3`，接口地址为 `https://dashscope.aliyuncs.com/compatible-mode/v1`。文档嵌入则使用前述 DashScope SDK，二者应分别核对。

原项目的系统提示词如下：

```python
SYSTEM_PROMPT = """你是一个专业的电商产品知识库助手。你的职责是基于提供的产品信息文档，准确回答用户关于产品的问题。

请严格遵守以下规则：
1. 只根据提供的【参考文档】内容回答问题，不要使用外部知识
2. 如果参考文档中没有相关信息，请诚实地说"根据现有产品资料，我无法找到相关信息"，不要编造
3. 回答时请引用具体的产品名称、规格参数，并在末尾标明引用来源编号
4. 如果涉及价格、库存等时效性信息，请提醒用户以实际页面为准
5. 使用中文回答，保持专业、友好的语气
6. 对于产品对比类问题，请用表格形式呈现差异

{context}"""
```

`_build_messages()` 先创建带参考资料的 `SystemMessage`，再加入近期历史消息，最后加入当前问题。当前源码截取 `chat_history[-10:]`，表示最多 10 条历史消息，不能直接解释成 10 轮问答；`config.py` 虽定义了 `CHAT_HISTORY_WINDOW`，该位置没有读取它。

系统提示词规定有资料时依据资料回答，没资料时承认缺失。它描述希望模型遵循的行为，需要通过问答验证；不能把写入“不要编造”当作已经保证所有回答正确。


## 13. 引用溯源：正文和来源必须能对上

原项目把相同的检索结果用于两件事：组织模型上下文，以及生成返回前端的 `sources`。`chain.py` 中的函数如下：

```python
def format_docs_with_sources(docs_with_scores: list) -> tuple:
    """Format retrieved docs into context string + sources list"""
    context_parts = []
    sources = []
    for i, (doc, score) in enumerate(docs_with_scores):
        source_label = f"[来源{i + 1}: {doc.metadata.get('filename', '未知文件')}]"
        context_parts.append(f"{source_label}\n{doc.page_content}")
        sources.append({
            "filename": doc.metadata.get("filename", "未知文件"),
            "chunk_text": doc.page_content[:300],
            "similarity_score": round(score, 4),
            "chunk_index": doc.metadata.get("chunk_index", 0),
            "product_category": doc.metadata.get("product_category", None),
        })
    return "\n\n".join(context_parts), sources
```

例如第一次遍历的片段在上下文中标为 `[来源1: 文件名]`，模型据此标注来源编号；前端得到对应来源列表。引用信息包含文件名、片段文本、相关分数、块序号和品类。

需要看清三个边界：

- 传给模型的是 `doc.page_content` 全部文本，返回给前端的 `chunk_text` 只截取前 **300 个字符**。因此 README 的“查看原文片段”应理解为当前返回的片段预览，不能说前端已得到整份文件或完整分块。
- 来源编号从 `1` 开始，入库时的 `chunk_index` 从 `0` 开始。二者是“本次检索的编号”和“文件中的块序号”，不要混用。
- `sources` 列表表示检索后提供给模型的参考片段，并没有逐句验证模型究竟使用了哪些内容。验收时还要对照回答、引用编号和片段正文。

历史消息同时保存回答正文和来源 JSON，才能在重新登录后继续查看当时返回的引用。只把引用临时留在浏览器内存，不能满足完整的历史恢复需求。


## 14. 多会话与 SSE：显示过程和保存过程要接起来

聊天接口是 `POST /api/conversations/{conv_id}/messages`。`api/chat.py` 的处理顺序是：先验证会话归属，读取近期历史，再保存当前用户消息，然后开始流式生成。

先读取历史的原因是避免把当前问题既放进历史、又作为当前消息追加一遍。历史读取默认取最近 10 条消息，再恢复为时间正序。首条提问还会用于会话标题，当前实现截取问题前 30 个字符，不是再调用一次模型生成标题。

SSE 是服务器发送事件，后端通过同一个 HTTP 响应持续向浏览器推送数据。当前链和 HTTP 层的字段有转换：

| 事件 | RAG 链内部数据 | HTTP SSE 的 `data` 内容 |
|---|---|---|
| `token` | `type="token"`、`content` | `{"token": "本次增量文本"}` |
| `sources` | `type="sources"`、`sources` | `{"sources": [...]}` |
| `done` | `type="done"`、`full_response` | `{"message_id": ..., "token_count": ...}` |
| `error` | HTTP 生成器捕获异常 | `{"error": "错误信息"}` |

下面是协议格式说明，**不是本次运行输出**。每个事件以空行结束：

```text
event: token
data: {"token": "增量文字"}

event: sources
data: {"sources": []}

```

`stream_rag_response()` 使用 `llm.astream(messages)` 逐步产生正文，生成结束后发送引用，再发完成事件。HTTP 生成器收到完成事件时，用独立的数据库会话保存助手消息及来源，提交后才返回包含 `message_id` 的 `done`。

流式显示解决“等待时能看到内容”的体验问题，持久化解决“以后能找回内容”的数据问题。页面看到文字，并不能单独证明消息已经成功保存；重新进入会话是必要的功能验收步骤。

当前 `token_count` 使用 `len(full_response)`，实际记录的是字符串长度，不是模型计费 token 数。模型输出的一个增量块也不必等于一个 token。阅读接口字段时应按代码确认含义。

前端接入时需要同时约定 POST 请求体、Bearer 令牌、事件名和 JSON 字段。浏览器原生 `EventSource` 的使用方式不能直接照搬到需要 POST 和自定义认证头的接口；本次没有取得前端流式客户端正文，不把某种客户端实现写成课程原代码。


## 15. 前端与接口：页面要连到真实后端

项目 `README.md` 列出的页面能力包括注册登录、我的会话、知识库管理、密码修改，以及 Markdown 回答和引用展示。`frontend/package.json` 确认使用 React、Ant Design、Zustand、Axios 和 `react-markdown`；`src/` 也按页面、组件、状态和请求等目录组织。

下面按实际路由及 README 整理首版联调入口：

| 方法与路径 | 用途 | 权限 |
|---|---|---|
| `POST /api/auth/register` | 注册 | 公开 |
| `POST /api/auth/login` | 登录 | 公开 |
| `GET /api/auth/me` | 获取当前用户 | 登录 |
| `PUT /api/auth/change-password` | 修改密码 | 登录 |
| `GET /api/conversations` | 当前用户会话列表 | 登录 |
| `POST /api/conversations` | 新建会话 | 登录 |
| `GET /api/conversations/{id}` | 读取会话及消息 | 所有者；快照中管理员也可读取 |
| `PATCH /api/conversations/{id}` | 修改标题 | 所有者 |
| `DELETE /api/conversations/{id}` | 软删除会话 | 所有者 |
| `POST /api/conversations/{id}/messages` | 流式知识问答 | 所有者 |
| `GET /api/kb/documents` | 文档列表 | 管理员 |
| `POST /api/kb/documents/upload` | 上传并安排索引 | 管理员 |
| `GET/DELETE /api/kb/documents/{id}` | 文档详情 / 删除 | 管理员 |
| `GET /api/kb/stats` | 文档和分块统计 | 管理员 |
| `GET /api/kb/search` | 直接检索，辅助核对片段 | 管理员 |
| `POST /api/kb/reindex` | 调用项目重建索引流程 | 管理员 |
| `GET /api/health` | 基础健康信息 | 公开 |

**原资料差异：**README 把上传概括为 `POST /api/kb/documents`，实际 `kb.py` 定义的是 `/documents/upload`，前端联调和接口调用应以后者为准。API 文档路径为 `/docs`，不要与健康接口混淆。

前端初版至少要把登录状态传给请求层，把会话 ID 传给聊天接口，并处理 `token`、`sources`、`done` 和 `error`。知识管理页需要在上传后继续查询状态，而不是上传返回就显示“已索引”。这些是基于已读取后端协议整理的联调要求；没有据此声称共享前端的每一项行为都已运行验证。

此外，快照的 `rebuild_index()` 虽在注释中写“清空”，实际先重置进程内实例、再向同一持久化集合添加分块，没有调用集合清空方法。不能把函数名称当作已经完成删除旧索引的证据，本篇也不要求首次验收先执行重建。


## 16. 从已讨论方案进入初版代码生成

本节的工作阶段是“方案讨论 → 初版生成”。原讲义保留了提出需求的提示词，共享源码提供了可核对的实现结果；中间完整对话和文件生成顺序没有保留。下面的步骤是据此整理的实施顺序。

1. **确定方案与接口。**明确模型、存储、账号角色、两条 RAG 流程、数据模型、接口和配置。
2. **生成后端骨架。**先建立 `main.py`、配置、数据库和请求响应结构，使服务入口和健康接口具备明确位置。
3. **实现身份与会话。**把注册登录、管理员依赖和会话归属连接起来，保存用户消息和历史。
4. **实现文档入库。**连接上传、状态记录、加载分块、嵌入和持久化向量库。
5. **实现检索生成与引用。**让同一批检索片段进入模型上下文和来源列表，再接入流式返回与消息保存。
6. **生成前端并联调。**页面接真实接口，处理权限、文档状态、聊天增量和引用。
7. **给出启动说明和首次验收路径。**列清配置、依赖、工作目录和真实待验证项。

**补充提示词：**方案确认后，可以用下面的表达限定初版交付范围。这不是课堂原文，也不代表本次整理正在执行这些开发动作。

```text
按我们已经确认的方案生成首版代码，完成注册登录、管理员知识库管理、
多用户多会话、历史持久化、RAG 问答和引用片段展示。
保持前后端接口路径、请求字段和流式事件一致，页面接入真实后端。
给出 .env.example、依赖清单、目录结构、启动命令和一条完整的使用路径。
每完成一个阶段说明生成或修改了哪些文件，还有哪些配置需要我填写。
没有真实验证的能力请标为待验证，先完成首版主流程，不展开后续压测。
```

AI 报告“代码已生成”之后，还要回到方案检查：接口是否真的存在，引用元数据是否从入库传到回答，历史消息是否落库，权限是否由后端验证。前文 README 与实际上传路由的差异，就是说明文档也需要对照源码的具体例子。

下一节再建立更完整的单元测试和 BUG 调试过程，本篇的首次验收先确认用户能走完整条业务流程。


## 17. 环境配置与依赖：先对照项目清单

项目 README 写明前置要求：Python `3.10+`、Node.js `18+`，以及可用的阿里云百炼 API Key。这里记录原项目要求；带有版本锁定的依赖仍需在所选解释器上实际安装验证，不能把版本下限理解成任意更高版本必然兼容。

在 `backend/` 配置 `.env`。下列示例根据已读取的 `config.py` 键名整理，**不是未能预览的 `.env.example` 原文**。API Key 和 JWT 密钥均须换成自己的值：

```dotenv
DASHSCOPE_API_KEY=sk-your-api-key-here
DASHSCOPE_API_BASE=https://dashscope.aliyuncs.com/compatible-mode/v1
LLM_MODEL=qwen-plus
EMBEDDING_MODEL=text-embedding-v2
LLM_TEMPERATURE=0.3

DATABASE_URL=sqlite+aiosqlite:///./data/app.db
CHROMA_PERSIST_DIR=./data/chromadb
CHROMA_COLLECTION_NAME=ecommerce_kb

JWT_SECRET=replace-with-your-random-secret
JWT_ALGORITHM=HS256
JWT_EXPIRE_MINUTES=1440

CHUNK_SIZE=800
CHUNK_OVERLAP=100
RETRIEVAL_TOP_K=5
RETRIEVAL_SCORE_THRESHOLD=0.4
CHAT_HISTORY_WINDOW=10

MAX_UPLOAD_SIZE_MB=50
ALLOWED_EXTENSIONS=txt,md,pdf,docx,csv,xlsx
CORS_ORIGINS=http://localhost:5173,http://localhost:3000
ADMIN_USERNAME=admin
ADMIN_PASSWORD=123456
```

`Settings` 从 `.env` 和环境变量加载配置。未写在 `.env` 中的字段使用代码默认值；已有进程环境变量也会影响最终配置。相对数据库与文件路径按启动工作目录解析，所以后端命令需要在 `backend/` 执行。

原 `requirements.txt` 的依赖如下，保留其锁定版本以便对应课程快照：

```text
fastapi==0.115.6
uvicorn[standard]==0.34.0
pydantic==2.10.3
pydantic-settings==2.7.0

sqlalchemy==2.0.36
aiosqlite==0.20.0
alembic==1.14.0

python-jose[cryptography]==3.3.0
passlib[bcrypt]==1.7.4
bcrypt==4.0.1
python-multipart==0.0.18

langchain==0.3.13
langchain-community==0.3.13
langchain-chroma==0.2.1
langchain-openai==0.3.0

dashscope==1.20.0

pypdf2==3.0.1
python-docx==1.1.2
openpyxl==3.1.5

chromadb==0.5.23

httpx==0.28.1
python-dotenv==1.0.1
```

**依赖与加载器要一起核对：**清单中的 `pypdf2` 与 `PyPDFLoader` 使用的 `pypdf` 不是同一个包；Word、Excel 路径采用的是 `Unstructured...Loader`，不能用清单里有 `python-docx`、`openpyxl` 就断言相关解析依赖已全部满足。先按实际加载器和报错补齐兼容依赖，再验证各格式，不把原依赖文件改写成未经运行的“保证可用版本”。

前端 `package.json` 的开发和构建脚本是：

```json
{
  "dev": "vite",
  "build": "tsc -b && vite build",
  "preview": "vite preview"
}
```

这是原文件 `scripts` 中与本篇相关的摘录，实际文件还包含测试脚本。`npm run dev` 启动开发服务，`npm run build` 先进行 TypeScript 构建再生成前端产物；两者的完成状态应分别报告。


## 18. 启动初版：工作目录和解释器要一致

README 提供 Windows `start.bat` 一键启动方式，说明脚本会准备后端虚拟环境与前端依赖，再启动 `8000` 和 `5173` 两个服务。本次只核对了脚本存在，没有取得其正文；以下手动启动步骤以已读取的 README 为依据。

### 后端：README 原命令

从项目根目录打开 Windows **cmd**，先进入 `backend/`：

```cmd
cd backend
python -m venv venv
venv\Scripts\activate
pip install -r requirements.txt
python -m uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
```

原 README 把这一组标为 Bash，但 `venv\Scripts\activate` 属于 Windows 路径写法，这里按 cmd 标注。激活后应确认安装与运行使用的是同一虚拟环境。

### 补充：Windows PowerShell 直接调用环境内解释器

仍从项目根目录进入 `backend/`。下面是原步骤的 PowerShell 写法，不依赖激活脚本；仅在本机访问时把监听地址设为 `127.0.0.1`。共享目录自带的 `venv/` 可能绑定原电脑路径，迁移到自己的电脑时应根据依赖清单重建环境。

```powershell
cd backend
python -m venv venv
.\venv\Scripts\python.exe -m pip install -r requirements.txt
.\venv\Scripts\python.exe -m uvicorn app.main:app --host 127.0.0.1 --port 8000 --reload
```

原命令的 `0.0.0.0` 监听全部网络接口；本机浏览器仍使用 `localhost` 或 `127.0.0.1` 访问。`--reload` 用于开发时监视代码变化。

### 前端：另开终端

后端终端保持运行，另开一个 PowerShell 终端，从项目根目录执行：

```powershell
cd frontend
npm install
npm run dev
```

README 预期前端位于 [http://localhost:5173](http://localhost:5173)，API 文档位于 [http://localhost:8000/docs](http://localhost:8000/docs)。如果开发服务实际使用其他端口，应核对前端请求配置和后端 `CORS_ORIGINS`，不能只按预期 URL 判断启动结果。

在另一个 PowerShell 终端读取基础健康接口：

```powershell
Invoke-RestMethod -Uri 'http://localhost:8000/api/health'
```

按 `system.py`，响应包含 `status`、`app`、`version`、`dashscope`、`chromadb`、`database`。其中 `dashscope="configured"` 只检查 API Key 字符串是否非空，没有真的请求模型；健康接口也没有执行数据库查询或向量检索。因此健康响应成功后，仍需完成实际上传与问答。


## 19. 首次功能验收：用课程商品资料走完整流程

原项目根目录 `data/products/` 提供 `electronics.md` 和 `clothing.md`。本次已读取 `electronics.md`，内容包括笔记本、手机和耳机的教学商品资料。以下问题依据这份文件设计，是**待执行的验收步骤，不是本次已通过的结果**；样例参数用于验证资料引用，不作为真实商品参数核实。

### 管理员上传与首次问答

1. 启动前后端，用课程管理员 `admin / 123456` 登录。
2. 进入知识库管理，上传 `data/products/electronics.md`；如需分类，可填写电子产品相关品类。
3. 等待文档状态变为 `indexed`，确认分块数量大于 0。若长期 `processing` 或报错，查看加载、嵌入和后台事务日志。
4. 进入“我的会话”，新建会话，提问：“索尼 WH-1000XM5 开启降噪后续航多久？”
5. 对照样例文件中的“降噪开启约30小时”，检查回答是否依据资料，并能显示包含相关文字的来源片段。
6. 追问资料未提供的信息，例如“这款耳机今天还剩多少库存？”，检查模型是否承认资料缺失，而不是编出库存数字。
7. 重新加载页面或退出后再登录，检查这次会话、问题、回答与引用是否仍能恢复。

由于引用预览只截取片段前 300 个字符，如果回答依据的参数落在更后面，预览可能无法显示完整依据。这时应区分“检索到了什么”“给模型传了什么”和“前端展示了什么”，不能仅凭出现一个文件名就完成引用验收。

### 普通用户与会话隔离

| 操作 | 需要观察的结果 |
|---|---|
| 注册一个普通用户并登录 | 能创建自己的会话并提问 |
| 普通用户查看知识管理入口 | 管理入口受到限制 |
| 普通用户直接请求管理 API | 后端返回权限拒绝；不能只验收菜单隐藏 |
| 同一用户创建第二个会话 | 两个会话的消息和历史上下文分别保存 |
| 使用另一普通用户读取或发送到已有会话 ID | 被后端拒绝访问其他用户会话 |
| 按旧密码、新密码流程修改密码 | 旧密码验证和后续登录行为符合实现 |

管理员读取会话详情的例外、管理员重启时重设配置密码的行为，按前文已核对的源码记录；不能用普通用户隔离的结果概括管理员所有权限。

完成这些操作，才能报告首版主流程是否可用。依赖安装成功、页面能够打开、健康接口返回正常、模型能说话，分别只是这条流程中的部分证据。单元测试、系统性 BUG 修复和并发性能测量继续放在后续课程中展开。


## 20. 来源与整理状态

课程来源：

- [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)：核对第 21 节标题及后续章节顺序。
- [Part3 LangChain RAG项目实战](https://www.yuque.com/xxcls/vibecoding/mtwqfc61tifo9cgi)：读取完整初始化提示词，保留“先讨论”的阶段约束。

网盘 `项目/LangChainRAG项目/` 中已读取的资料：

| 文件或目录 | 本篇使用的内容 |
|---|---|
| `README.md` | 技术栈、功能、启动步骤、结构和接口概览 |
| `backend/requirements.txt` | 后端依赖与锁定版本 |
| `backend/app/config.py` | 模型、路径、RAG、权限及上传默认参数 |
| `backend/app/main.py`、`database.py` | 启动生命周期、路由、数据库会话与初始化 |
| `backend/app/models/` 四个主要模型 | 用户、会话、消息和知识文档的数据关系 |
| `backend/app/core/dependencies.py` | 令牌解析、当前用户与管理员检查 |
| `backend/app/services/` 四个服务文件 | 认证、会话操作、知识库管理和摄入编排 |
| `backend/app/rag/` 六个实现文件 | 加载、分块、嵌入、向量存储、检索、提示词和引用 |
| `backend/app/api/chat.py`、`kb.py`、`system.py` | 实际上传路由、SSE、后台入库和健康接口 |
| `frontend/package.json` | 前端依赖与脚本 |
| `data/products/electronics.md` | 首次功能验收的样例依据 |

`.env.example`、`start.bat` 只核对存在，未取得正文；前端 TypeScript 页面和请求实现未逐文件读取。`docs/` 页面未提供可读取的方案正文，因此没有把补充计划写成课堂的实际讨论回复。

本次只新增 10 Notebook，没有修改 01—09 笔记，没有运行课程项目、安装依赖、调用付费模型、创建业务账号或执行 Git 提交。文中的代码用于阅读，启动和验收步骤供后续实践；格式校验不等于这些业务步骤已经执行通过。资料核对日期：`2026-10-10`。
